# 02 — CA Gubernatorial Race 2026: Donation Distribution & OpenSecrets Matching

This notebook visualises the distribution of campaign contributions in the 2026 CA Governor's race and matches individual donors against the OpenSecrets contribution database.

**Three donor categories** (per CalAccess/FPPC logic):
| Donor Type | Rule |
|---|---|
| **Individual** | Has a `Contributor Occupation` entry |
| **Organization** | No Contributor ID AND no Contributor Occupation |
| **PAC / Committee** | Has a `Contributor ID` |

**Four individual spending tiers** (based on CA FPPC individual contribution limits):
| Tier | Range |
|---|---|
| High Spender | ≥ $36,400 (FPPC statewide limit) |
| Medium Spender | $10,000 – $36,400 |
| Lower Spender | $200 – $10,000 |
| Everyday | < $200 |

---


In [ ]:
from __future__ import annotations
import re, textwrap, warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
%matplotlib inline
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
})

# ── Paths ──────────────────────────────────────────────────────────────────────
# Adjust if needed. BASE_DIR is the root of this project folder.
BASE_DIR        = Path("..").resolve()          # one level up from code/
DATA_DIR        = BASE_DIR / "data"
GOV_RACE_PATH   = DATA_DIR / "01CalAccess_CampaignFinance_Data" / "governor_race_2026-03-20.csv"
OPENSECRETS_PATH = DATA_DIR / "ContributionOpenSecretsCategories (1).csv"

OUTPUT_DIR = DATA_DIR / "02_output"
FIG_DIR    = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ── Constants ──────────────────────────────────────────────────────────────────
CA_INDIVIDUAL_LIMIT = 36_400   # FPPC 2025-26 statewide race individual limit

BUCKET_LIMITS = {
    "Everyday (<$200)":     (0,       200),
    "Lower ($200-$10K)":    (200,   10_000),
    "Medium ($10K-limit)":  (10_000, CA_INDIVIDUAL_LIMIT),
    "High (≥ limit)":       (CA_INDIVIDUAL_LIMIT, float("inf")),
}
BUCKET_ORDER   = list(BUCKET_LIMITS.keys())
BUCKET_COLORS  = ["#74B9E1", "#55A868", "#E8A838", "#C44E52"]

DONOR_COLORS = {
    "Individual":    "#4C72B0",
    "Organization":  "#55A868",
    "PAC/Committee": "#C44E52",
}

print("Paths configured. Data directory:", DATA_DIR)


In [ ]:
# ── Helper utilities ───────────────────────────────────────────────────────────

def clean_amount(series: pd.Series) -> pd.Series:
    """Strip dollar signs / commas and coerce to float."""
    s = series.astype(str).str.replace(r"[$,]", "", regex=True).str.strip()
    return pd.to_numeric(s, errors="coerce")


def detect_col(df: pd.DataFrame, candidates: list[str]) -> str | None:
    """Return the first candidate column name that exists (case-insensitive)."""
    low = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in low:
            return low[c.lower()]
    return None


def normalise_name(name: str) -> str:
    """Upper-case, strip punctuation, collapse whitespace for matching."""
    if not isinstance(name, str):
        return ""
    n = re.sub(r"[^A-Z0-9 ]", " ", name.upper())
    return re.sub(r"\s+", " ", n).strip()


## 1. Load Data & Classify Donor Types

In [ ]:
df = pd.read_csv(GOV_RACE_PATH, dtype=str, low_memory=False)
print(f"{len(df):,} rows × {df.shape[1]} columns")
print("Columns:", list(df.columns))


In [ ]:
# ── Locate key columns ────────────────────────────────────────────────────────
col_name = detect_col(df, ["Contributor Name", "CTRIB_NAML", "contributor_name"])
col_occ  = detect_col(df, ["Contributor Occupation", "CTRIB_OCC"])
col_id   = detect_col(df, ["Contributor ID", "CMTE_ID", "FILER_ID", "Committee ID"])
col_amt  = detect_col(df, ["Amount", "AMOUNT", "TRAN_AMT1"])

print(f"Name col   : {col_name}")
print(f"Occupation : {col_occ}")
print(f"ID col     : {col_id}")
print(f"Amount col : {col_amt}")


In [ ]:
# ── Classify each row as Individual / Organization / PAC/Committee ────────────
df["amount_clean"] = clean_amount(df[col_amt])

# Boolean masks
has_occ = (
    df[col_occ].notna()
    & df[col_occ].str.strip().ne("")
    & df[col_occ].str.lower().ne("nan")
) if col_occ else pd.Series(False, index=df.index)

has_id = (
    df[col_id].notna()
    & df[col_id].str.strip().ne("")
    & df[col_id].str.lower().ne("nan")
) if col_id else pd.Series(False, index=df.index)

df["donor_type"] = np.select(
    [has_id, has_occ],
    ["PAC/Committee", "Individual"],
    default="Organization",
)

# Convenience columns
df["_name"] = df[col_name].fillna("").str.strip()
df["_occ"]  = df[col_occ].fillna("").str.strip() if col_occ else ""

# Summary
counts = df["donor_type"].value_counts()
pcts   = counts / len(df) * 100
summary = pd.DataFrame({"Count": counts, "Pct (%)": pcts.round(2)})
print("\nDonor type breakdown:\n")
display(summary)


## 2. Distribution of Donation Amounts by Donor Type

Each panel below shows the **histogram of donation sizes** for one donor type,
with a log-scaled x-axis so we can see both tiny ($1) and very large ($36K+) donations
in the same view. The dashed line marks the **median**.


In [ ]:
donor_types = ["Individual", "Organization", "PAC/Committee"]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(
    "Distribution of Contribution Amounts by Donor Type\n(CA Gubernatorial Race 2026)",
    fontsize=14, y=1.03,
)

for ax, dt in zip(axes, donor_types):
    sub = df.loc[df["donor_type"] == dt, "amount_clean"].dropna()
    sub = sub[sub > 0]

    if sub.empty:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(dt)
        continue

    log_min = max(0, np.floor(np.log10(sub.min())))
    log_max = np.ceil(np.log10(sub.max()))
    bins = np.logspace(log_min, log_max, 40)

    ax.hist(sub, bins=bins, color=DONOR_COLORS[dt], edgecolor="white",
            linewidth=0.5, alpha=0.88)
    ax.set_xscale("log")
    ax.xaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:,.0f}" if x >= 1 else f"${x:.2f}")
    )

    # Median line
    med = sub.median()
    ax.axvline(med, color="black", linestyle="--", linewidth=1.3, alpha=0.75)
    ax.text(
        med * 1.2, ax.get_ylim()[1] * 0.88,
        f"Median\n${med:,.0f}", fontsize=8, color="black", va="top",
    )

    ax.set_title(f"{dt}\n(n = {len(sub):,})", fontsize=12)
    ax.set_xlabel("Donation Amount (log scale)")
    ax.set_ylabel("# of Contributions")
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

fig.tight_layout()
fig.savefig(FIG_DIR / "01_donation_distributions.png", bbox_inches="tight")
plt.show()
print("Saved: 01_donation_distributions.png")


In [ ]:
# Key statistics per donor type
stats_rows = []
for dt in donor_types:
    sub = df.loc[df["donor_type"] == dt, "amount_clean"].dropna()
    sub_pos = sub[sub > 0]
    stats_rows.append({
        "Donor Type":     dt,
        "# Contributions": f"{len(sub):,}",
        "Total ($)":       f"${sub.sum():,.0f}",
        "Mean ($)":        f"${sub.mean():,.0f}",
        "Median ($)":      f"${sub.median():,.0f}",
        "Max ($)":         f"${sub.max():,.0f}",
        "Min ($)":         f"${sub_pos.min():,.2f}" if not sub_pos.empty else "-",
    })

display(pd.DataFrame(stats_rows).set_index("Donor Type"))


## 3. Categorising Individual Donors into Spending Tiers

We split individual donors into four buckets based on the CA FPPC individual 
contribution limit (~$36,400 for 2025-26 statewide races):

| Tier | Range |
|---|---|
| **High Spender** | ≥ FPPC limit (~$36,400) |
| **Medium Spender** | $10,000 – limit |
| **Lower Spender** | $200 – $10,000 |
| **Everyday** | < $200 |


In [ ]:
indiv = df[df["donor_type"] == "Individual"].copy()

def assign_bucket(amt):
    for label, (lo, hi) in BUCKET_LIMITS.items():
        if lo <= amt < hi:
            return label
    return BUCKET_ORDER[-1]

indiv["bucket"] = indiv["amount_clean"].apply(
    lambda x: assign_bucket(x) if pd.notna(x) else "Everyday (<$200)"
)

bucket_stats = (
    indiv.groupby("bucket", observed=True)["amount_clean"]
    .agg(n="count", total="sum", mean="mean", median="median")
    .reindex(BUCKET_ORDER)
    .fillna(0)
)
bucket_stats["pct_of_donors"] = (bucket_stats["n"] / len(indiv) * 100).round(2)
bucket_stats["pct_of_dollars"] = (bucket_stats["total"] / indiv["amount_clean"].sum() * 100).round(2)

print(f"Total individual contributions: {len(indiv):,}")
print(f"CA FPPC individual limit: ${CA_INDIVIDUAL_LIMIT:,}\n")
display(bucket_stats.style.format({
    "n": "{:,.0f}", "total": "${:,.0f}", "mean": "${:,.0f}",
    "median": "${:,.0f}", "pct_of_donors": "{:.1f}%", "pct_of_dollars": "{:.1f}%",
}))


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(
    "Individual Donors by Spending Tier\n(CA Gubernatorial Race 2026)",
    fontsize=14,
)
x = np.arange(len(BUCKET_ORDER))
labels_wrap = [textwrap.fill(b, 14) for b in BUCKET_ORDER]

# Panel A: count
bars1 = ax1.bar(x, bucket_stats["n"], color=BUCKET_COLORS, edgecolor="white")
ax1.set_xticks(x); ax1.set_xticklabels(labels_wrap, fontsize=9)
ax1.set_ylabel("Number of Contributions")
ax1.set_title("Count of Individual Contributions")
for bar, v in zip(bars1, bucket_stats["n"]):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
             f"{v:,.0f}", ha="center", va="bottom", fontsize=8)

# Panel B: total dollars
bars2 = ax2.bar(x, bucket_stats["total"]/1e6, color=BUCKET_COLORS, edgecolor="white")
ax2.set_xticks(x); ax2.set_xticklabels(labels_wrap, fontsize=9)
ax2.set_ylabel("Total ($ millions)")
ax2.set_title("Total Dollars per Tier")
for bar, v in zip(bars2, bucket_stats["total"]/1e6):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f"${v:.2f}M", ha="center", va="bottom", fontsize=8)

fig.text(
    0.5, -0.04,
    f"High Spender threshold = CA FPPC individual limit ≈ ${CA_INDIVIDUAL_LIMIT:,} (2025-26).",
    ha="center", fontsize=8, style="italic", color="gray",
)
fig.tight_layout()
fig.savefig(FIG_DIR / "02_individual_buckets.png", bbox_inches="tight")
plt.show()
print("Saved: 02_individual_buckets.png")


### 3a. Deep Dive: Who are the High Spenders?

Let's look at the top donors in the "High Spender" tier — names and amounts.


In [ ]:
high = indiv[indiv["bucket"] == "High (≥ limit)"].copy()
high_summary = (
    high.groupby("_name")["amount_clean"]
    .agg(n_contributions="count", total_donated="sum")
    .sort_values("total_donated", ascending=False)
    .head(30)
)
high_summary.index.name = "Contributor Name"
display(high_summary.style.format({"total_donated": "${:,.0f}", "n_contributions": "{:,.0f}"}))


## 4. OpenSecrets Name Matching

We attempt to match individual donor names against the 
**ContributionOpenSecretsCategories** database (~298K entries, each with an 
industry classification at three levels of specificity).

**Matching strategy**: case-insensitive exact match after normalising punctuation 
and whitespace. This is conservative — fuzzy matching would increase recall but 
at the cost of precision.

**Key question**: Do high-dollar donors (especially the "High Spender" tier) 
appear in OpenSecrets, indicating they are repeat political donors with known 
industry affiliations?


In [ ]:
os_df = pd.read_csv(OPENSECRETS_PATH, dtype=str, low_memory=False)
os_df.columns = [c.strip() for c in os_df.columns]
os_df["name_norm"] = os_df["name"].apply(normalise_name)

print(f"OpenSecrets entries: {len(os_df):,}")
print(f"  Non-Uncoded (has industry): {(os_df['level1_category'] != 'Uncoded').sum():,}")
print()
print("Top level1 categories:")
display(os_df["level1_category"].value_counts().head(10).to_frame("count"))


In [ ]:
# Build lookup dict: normalised name → row data
os_lookup = (
    os_df.drop_duplicates("name_norm")
    .set_index("name_norm")[["level1_category", "level2_category", "level3_category"]]
    .to_dict(orient="index")
)

# Match
indiv["name_norm"]  = indiv["_name"].apply(normalise_name)
indiv["os_matched"] = indiv["name_norm"].isin(os_lookup)
indiv["os_level1"]  = indiv["name_norm"].map({k: v["level1_category"] for k, v in os_lookup.items()})
indiv["os_level2"]  = indiv["name_norm"].map({k: v["level2_category"] for k, v in os_lookup.items()})
indiv["os_level3"]  = indiv["name_norm"].map({k: v["level3_category"] for k, v in os_lookup.items()})

print(f"Overall individual match rate: {indiv['os_matched'].mean()*100:.1f}%")


### 4a. Match Rate by Spending Tier

In [ ]:
match_rows = []
for b in BUCKET_ORDER:
    sub = indiv[indiv["bucket"] == b]
    total    = len(sub)
    matched  = sub["os_matched"].sum()
    with_ind = (sub["os_matched"] & sub["os_level1"].ne("Uncoded")).sum()
    match_rows.append({
        "Tier":                     b,
        "N Contributions":          total,
        "In OpenSecrets":           int(matched),
        "Match Rate":               f"{matched/total*100:.1f}%" if total else "—",
        "Has Industry (non-Uncoded)": int(with_ind),
        "Industry Match Rate":      f"{with_ind/total*100:.1f}%" if total else "—",
    })

match_table = pd.DataFrame(match_rows).set_index("Tier")
display(match_table)


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))

stack_data = []
for b in BUCKET_ORDER:
    sub = indiv[indiv["bucket"] == b]
    n = len(sub)
    has_ind  = (sub["os_matched"] & sub["os_level1"].ne("Uncoded")).sum()
    has_unc  = (sub["os_matched"] & sub["os_level1"].eq("Uncoded")).sum()
    no_match = (~sub["os_matched"]).sum()
    stack_data.append({"Has Industry": has_ind, "In OS (Uncoded)": has_unc,
                        "Not in OpenSecrets": no_match, "total": n})

stack_df = pd.DataFrame(stack_data, index=BUCKET_ORDER)
stack_colors = ["#4C72B0", "#74B9E1", "#CCCCCC"]
cols = ["Has Industry", "In OS (Uncoded)", "Not in OpenSecrets"]

bottom = np.zeros(len(BUCKET_ORDER))
for col, color in zip(cols, stack_colors):
    vals = stack_df[col].values.astype(float)
    bars = ax.bar(BUCKET_ORDER, vals, bottom=bottom, color=color,
                  edgecolor="white", linewidth=0.5, label=col)
    for bar, v in zip(bars, vals):
        if v > stack_df["total"].max() * 0.025:
            yc = bar.get_y() + bar.get_height() / 2
            ax.text(bar.get_x() + bar.get_width()/2, yc, f"{int(v):,}",
                    ha="center", va="center", fontsize=8, color="white", fontweight="bold")
    bottom += vals

ax.set_xticks(range(len(BUCKET_ORDER)))
ax.set_xticklabels([textwrap.fill(b, 14) for b in BUCKET_ORDER], fontsize=9)
ax.set_ylabel("Number of Contributions")
ax.set_title(
    "OpenSecrets Match Rate by Individual Donor Tier\n"
    "(CA Gubernatorial Race 2026)", fontsize=13,
)
ax.legend(loc="upper right", frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()
fig.savefig(FIG_DIR / "03_opensecrets_match_rates.png", bbox_inches="tight")
plt.show()
print("Saved: 03_opensecrets_match_rates.png")


### 4b. Industry Sector Breakdown (Matched Donors with Known Industry)

In [ ]:
matched_ind = indiv[indiv["os_matched"] & indiv["os_level1"].ne("Uncoded")].copy()

if matched_ind.empty:
    print("No donors with confirmed industry classifications — check match quality above.")
else:
    pivot = (
        matched_ind.groupby(["bucket", "os_level1"])
        .size()
        .unstack(fill_value=0)
        .reindex(BUCKET_ORDER)
        .fillna(0)
    )

    # Keep top 10 sectors + collapse rest to "Other"
    TOP_N = 10
    top_sectors = pivot.sum().nlargest(TOP_N).index.tolist()
    other_col = pivot.drop(columns=top_sectors, errors="ignore").sum(axis=1)
    pivot = pivot[top_sectors].copy()
    if (other_col > 0).any():
        pivot["Other"] = other_col
        top_sectors.append("Other")

    # As % share within each bucket
    pivot_pct = pivot.div(pivot.sum(axis=1), axis=0).fillna(0) * 100

    fig, ax = plt.subplots(figsize=(13, 5.5))
    cmap = plt.get_cmap("tab20", len(top_sectors))
    bottom = np.zeros(len(BUCKET_ORDER))

    for i, col in enumerate(top_sectors):
        vals = pivot_pct[col].values if col in pivot_pct.columns else np.zeros(len(BUCKET_ORDER))
        ax.bar(BUCKET_ORDER, vals, bottom=bottom, color=cmap(i),
               edgecolor="white", linewidth=0.4, label=col)
        bottom += vals

    count_per_bucket = matched_ind.groupby("bucket").size().reindex(BUCKET_ORDER).fillna(0)
    for i, b in enumerate(BUCKET_ORDER):
        n = int(count_per_bucket[b])
        ax.text(i, 102, f"n={n:,}", ha="center", va="bottom", fontsize=8, color="gray")

    ax.set_xticks(range(len(BUCKET_ORDER)))
    ax.set_xticklabels([textwrap.fill(b, 14) for b in BUCKET_ORDER], fontsize=9)
    ax.set_ylim(0, 108)
    ax.set_ylabel("Share of Industry-Matched Donors (%)")
    ax.set_title(
        "Industry Sector Mix — OpenSecrets-Matched Individual Donors\n"
        "(CA Gubernatorial Race 2026)", fontsize=13,
    )
    ax.legend(loc="upper right", bbox_to_anchor=(1.22, 1), frameon=False, fontsize=8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    fig.tight_layout()
    fig.savefig(FIG_DIR / "04_matched_industry_by_bucket.png", bbox_inches="tight")
    plt.show()
    print("Saved: 04_matched_industry_by_bucket.png")


### 4c. High Spenders — Detailed Industry View

For the "High Spender" tier specifically, let's see the level-2 and level-3 
industry breakdowns (more granular than the sector view above).


In [ ]:
high_matched = indiv[
    (indiv["bucket"] == "High (≥ limit)")
    & indiv["os_matched"]
    & indiv["os_level1"].ne("Uncoded")
].copy()

print(f"High Spenders with confirmed industry: {len(high_matched):,}")

if not high_matched.empty:
    print("\n── Level 1 (Sector) ──")
    display(high_matched["os_level1"].value_counts().head(15).to_frame("count"))
    print("\n── Level 2 (Sub-sector) ──")
    display(high_matched["os_level2"].value_counts().head(15).to_frame("count"))
    print("\n── Level 3 (Specific industry) ──")
    display(high_matched["os_level3"].value_counts().head(15).to_frame("count"))


## 5. Save Summary CSV

In [ ]:
rows = []
for b in BUCKET_ORDER:
    sub = indiv[indiv["bucket"] == b]
    n    = len(sub)
    tot  = sub["amount_clean"].sum()
    mat  = sub["os_matched"].sum()
    ind  = (sub["os_matched"] & sub["os_level1"].ne("Uncoded")).sum()
    rows.append({
        "bucket":                  b,
        "n_contributions":         n,
        "total_amount_usd":        round(tot, 2),
        "n_matched_opensecrets":   int(mat),
        "pct_matched":             round(mat / n * 100, 2) if n else 0,
        "n_with_industry":         int(ind),
        "pct_with_industry":       round(ind / n * 100, 2) if n else 0,
    })

summary_csv = pd.DataFrame(rows)
out_path = OUTPUT_DIR / "opensecrets_match_summary.csv"
summary_csv.to_csv(out_path, index=False)
print("Saved:", out_path)
display(summary_csv)


## 6. Interpretation & Key Takeaways

Run the cells above first, then fill in the numbers below.

**Hypothesis**: High-dollar individual donors ($36,400+) are more likely to be 
repeat political donors who appear in OpenSecrets, and can therefore be assigned 
an industry classification.

**What to look for**:
- **Match rate vs. spending tier**: Does the match rate increase as donation size grows?
- **Industry of high spenders**: Are certain sectors (Finance, Real Estate, Lawyers, 
  Energy) disproportionately represented in the high-spend tier vs. the everyday tier?
- **Volume vs. value**: The "Everyday" tier likely contains the most *donors*, 
  but the "High Spender" and "Medium Spender" tiers likely contribute the bulk 
  of *total dollars*.

---
*Script version of this notebook: `02_donation_analysis.py`  
Output figures: `data/02_output/figures/`*
